# Props Data Organizer

## Definitions of Formulas Used in This Notebook

### Estimated Value (EV)
EV is the average amount you can expect to win or lose per bet if you placed the same bet many times. It helps identify profitable betting opportunities by comparing the expected return to the risk involved.

**Formula:**
$$
\text{EV} = (\text{Probability of Winning} \times \text{Profit if Win}) - (\text{Probability of Losing} \times \text{Loss if Lose})
$$

### Kelly Criterion
The Kelly Criterion is a formula used to determine the optimal size of a series of bets. It aims to maximize the logarithm of wealth, balancing the trade-off between risk and reward. The formula considers both the probability of winning and the odds offered, guiding you on how much of your bankroll to wager on each bet.

**Formula:**
$$
\text{Kelly Fraction} = \frac{(\text{Probability of Winning} \times (\text{Odds} + 1)) - 1}{\text{Odds}}
$$

### Variance
Variance in sports betting represents the spread or dispersion of actual outcomes around the expected value. It's a crucial metric for understanding the risk and volatility associated with betting predictions. Higher variance indicates more volatile and unpredictable outcomes, while lower variance suggests more consistent results.

**Formula:**
$$
\text{Variance} = \frac{\sum_{i=1}^{n} (x_i - \mu)^2}{n}
$$

Where:
- $x_i$ represents each individual outcome
- $\mu$ is the mean or expected value
- $n$ is the total number of observations

In the context of prop betting:
- High variance props (e.g., 3-pointers made) tend to be more risky but potentially more profitable
- Low variance props (e.g., minutes played) typically offer more consistent but lower returns


In [7]:
import pandas as pd 
import numpy as np
import time
import requests
import os
import sys
from datetime import datetime
import joblib

# feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
if feature_path not in sys.path:
    sys.path.append(feature_path)
    
from PROPS_EV.calculateEVS import *
from MODELS.model import *

today = datetime.now()
formatted_date = today.strftime("%m_%d_%y")
pd.set_option('display.max_columns', None)

### Grabs players odds for the day (US all boookmakers, DFS is prizepicks and underdogs)

In [2]:
# from NBAPropFinder.NBAPropFinder import NBAPropFinder

# nba_props = NBAPropFinder(region='us_dfs')
# prizePicks = nba_props.dataframe
# prizePicks.head(10)

### Single Bets from bookmakers that dont include prizePicks or UnderDogs

In [6]:
features = [
    # Player context
    'PLAYER_ID', 'TEAM_ID', 'OPP_TEAM_ID', 
    'STARTING', 'HOME_GAME', 
    'PLAYER_DAYS_REST', 'IS_BACK_TO_BACK', 
    
    # Player season averages
    'PTS_AVG_TO_DATE', 'MIN_AVG_TO_DATE', 'FGA_AVG_TO_DATE', 'FTA_AVG_TO_DATE', 'FG3A_AVG_TO_DATE', 
    'FG_PCT_AVG_TO_DATE', 'FG3_PCT_AVG_TO_DATE', 'FT_PCT_AVG_TO_DATE', 'USG_PCT_AVG_TO_DATE', 'TS_PCT_AVG_TO_DATE', 
    'EFG_PCT_AVG_TO_DATE', 'POSS_AVG_TO_DATE', 'TCHS_AVG_TO_DATE', 'AST_AVG_TO_DATE', 'REB_AVG_TO_DATE', 'TOV_AVG_TO_DATE',
    
    # LAG
    'PTS_LAG_1', 'PTS_LAG_2', 'MIN_LAG_1', 'MIN_LAG_2', 'FGA_LAG_1', 'FGA_LAG_2', 'FTA_LAG_1', 'FTA_LAG_2', 'FG3A_LAG_1', 
    'FG3A_LAG_2', 'FG_PCT_LAG_1', 'FG_PCT_LAG_2', 'FG3_PCT_LAG_1', 'FG3_PCT_LAG_2', 'FT_PCT_LAG_1', 'FT_PCT_LAG_2', 
    'USG_PCT_LAG_1', 'USG_PCT_LAG_2', 'TS_PCT_LAG_1', 'TS_PCT_LAG_2', 'EFG_PCT_LAG_1', 'EFG_PCT_LAG_2', 'POSS_LAG_1', 'POSS_LAG_2', 
    'TCHS_LAG_1', 'TCHS_LAG_2', 'AST_LAG_1', 'AST_LAG_2', 'REB_LAG_1', 'REB_LAG_2', 'TOV_LAG_1', 'TOV_LAG_2',
    
    # 3 game rolling averages
    'PTS_ROLLING_AVG_3', 'MIN_ROLLING_AVG_3', 'FGA_ROLLING_AVG_3', 'FTA_ROLLING_AVG_3', 'FG3A_ROLLING_AVG_3', 'FG_PCT_ROLLING_AVG_3', 
    'FG3_PCT_ROLLING_AVG_3', 'FT_PCT_ROLLING_AVG_3', 'USG_PCT_ROLLING_AVG_3', 'TS_PCT_ROLLING_AVG_3', 'EFG_PCT_ROLLING_AVG_3', 
    'POSS_ROLLING_AVG_3', 'TCHS_ROLLING_AVG_3', 'AST_ROLLING_AVG_3', 'REB_ROLLING_AVG_3', 'TOV_ROLLING_AVG_3',

    # Short-term form (5-game rolling averages)
    'PTS_ROLLING_AVG_5', 'MIN_ROLLING_AVG_5', 'FGA_ROLLING_AVG_5', 'FTA_ROLLING_AVG_5', 'FG3A_ROLLING_AVG_5', 'FG_PCT_ROLLING_AVG_5', 
    'FG3_PCT_ROLLING_AVG_5', 'FT_PCT_ROLLING_AVG_5', 'USG_PCT_ROLLING_AVG_5', 'TS_PCT_ROLLING_AVG_5', 'EFG_PCT_ROLLING_AVG_5', 
    'POSS_ROLLING_AVG_5', 'TCHS_ROLLING_AVG_5', 'AST_ROLLING_AVG_5', 'REB_ROLLING_AVG_5', 'TOV_ROLLING_AVG_5',
    
    # 7 game rolling averages
    'PTS_ROLLING_AVG_7', 'MIN_ROLLING_AVG_7', 'FGA_ROLLING_AVG_7', 'FTA_ROLLING_AVG_7', 'FG3A_ROLLING_AVG_7', 'FG_PCT_ROLLING_AVG_7', 
    'FG3_PCT_ROLLING_AVG_7', 'FT_PCT_ROLLING_AVG_7', 'USG_PCT_ROLLING_AVG_7', 'TS_PCT_ROLLING_AVG_7', 'EFG_PCT_ROLLING_AVG_7', 
    'POSS_ROLLING_AVG_7', 'TCHS_ROLLING_AVG_7', 'AST_ROLLING_AVG_7', 'REB_ROLLING_AVG_7', 'TOV_ROLLING_AVG_7',

    # Medium-term form (15-game rolling averages)
    'PTS_ROLLING_AVG_15', 'MIN_ROLLING_AVG_15', 'FGA_ROLLING_AVG_15', 'FTA_ROLLING_AVG_15', 'FG3A_ROLLING_AVG_15', 'FG_PCT_ROLLING_AVG_15', 
    'FG3_PCT_ROLLING_AVG_15', 'FT_PCT_ROLLING_AVG_15', 'USG_PCT_ROLLING_AVG_15', 'TS_PCT_ROLLING_AVG_15', 'EFG_PCT_ROLLING_AVG_15', 
    'POSS_ROLLING_AVG_15', 'TCHS_ROLLING_AVG_15', 'AST_ROLLING_AVG_15', 'REB_ROLLING_AVG_15', 'TOV_ROLLING_AVG_15',
    
    # Long-term form (40-game rolling averages)
    'PTS_ROLLING_AVG_40', 'MIN_ROLLING_AVG_40', 'FGA_ROLLING_AVG_40', 'FTA_ROLLING_AVG_40', 'FG3A_ROLLING_AVG_40', 'FG_PCT_ROLLING_AVG_40', 
    'FG3_PCT_ROLLING_AVG_40', 'FT_PCT_ROLLING_AVG_40', 'USG_PCT_ROLLING_AVG_40', 'TS_PCT_ROLLING_AVG_40', 'EFG_PCT_ROLLING_AVG_40', 
    'POSS_ROLLING_AVG_40', 'TCHS_ROLLING_AVG_40', 'AST_ROLLING_AVG_40', 'REB_ROLLING_AVG_40', 'TOV_ROLLING_AVG_40',

    # Opponent
    'OPP_DEF_RATING_AVG_TO_DATE', 'OPP_PACE_AVG_TO_DATE', 'OPP_PTS_AVG_TO_DATE', 'OPP_FGA_AVG_TO_DATE', 
    'OPP_REB_AVG_TO_DATE', 'OPP_AST_AVG_TO_DATE', 'OPP_TOV_AVG_TO_DATE', 'OPP_BLK_AVG_TO_DATE', 'OPP_STL_AVG_TO_DATE',
    
    #starters 
    'TEAM_OFF_RATING_AVG_TO_DATE','TEAM_DEF_RATING_AVG_TO_DATE','TEAM_PACE_AVG_TO_DATE', 'TEAM_FGA_AVG_TO_DATE',
    'TEAM_PTS_AVG_TO_DATE', 'TEAM_REB_AVG_TO_DATE', 'TEAM_AST_AVG_TO_DATE', 'TEAM_TOV_AVG_TO_DATE',
    
    # Team odds
    'team_spread', 'total', 'team_is_favored','TEAM_IMPLIED_PTS_FAV','TEAM_IMPLIED_PTS_UND','BLOWOUT_RISK'
]

model = joblib.load('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/MODELS/Models/PTS_cat_model.pkl')
# model = joblib.load(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\MODELS\Models\PTS_cat_model.pkl")
data = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values('GAME_DATE', ascending=False)
bookmakers = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/singleBookies.csv')
prizePicks = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/odds25.csv')

date = '2024-12-11'
espnDate = '20241211'
odds = bookmakers[
    (bookmakers['CATEGORY'] == 'points') &
    (bookmakers['GAME_DATE'] == date) &
    (bookmakers['ODDS'] < 200) &
    (bookmakers['ODDS'] > -200)
]

oddsPP = prizePicks[
    (prizePicks['CATEGORY'] == 'player_points') &
    (prizePicks['GAME_DATE'] == date) &
    (prizePicks['BOOKMAKER'] == 'underdog')
]
oddsPP.rename(columns={'OVER/UNDER': 'SIDE'}, inplace=True)
filterData = data[data['GAME_DATE'] <= date].sort_values('GAME_DATE', ascending=True)
games = get_espn_games(date_str=espnDate)
oddsPP.head()

/var/folders/nw/9w4r5hrd05s122kt5hg_t8bw0000gn/T/ipykernel_92480/1204719561.py:59: DtypeWarning: Columns (10,11,13) have mixed types. Specify dtype option on import or set low_memory=False.
  prizePicks = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/odds25.csv')
/var/folders/nw/9w4r5hrd05s122kt5hg_t8bw0000gn/T/ipykernel_92480/1204719561.py:75: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  oddsPP.rename(columns={'OVER/UNDER': 'SIDE'}, inplace=True)


,Unnamed: 0.1,Unnamed: 0,NAME,CATEGORY,BOOKMAKER,SIDE,LINE,PRICE,HOME_TEAM,AWAY_TEAM,game_id,commence_time,GAME_DATE,period_id,fair_line,fair_odds,OVER/ODDS
25312,25312,25312,Bogdan Bogdanovic,player_points,underdog,Over,13.5,-137,New York Knicks,Atlanta Hawks,0667c1e4b0ffd20ab6acdceee432183f,2024-12-12T00:00:00Z,2024-12-11,NaN,NaN,NaN,over
25314,25314,25314,Trae Young,player_points,underdog,Over,21.5,-137,New York Knicks,Atlanta Hawks,0667c1e4b0ffd20ab6acdceee432183f,2024-12-12T00:00:00Z,2024-12-11,NaN,NaN,NaN,over
25315,25315,25315,Trae Young,player_points,underdog,Under,21.5,-137,New York Knicks,Atlanta Hawks,0667c1e4b0ffd20ab6acdceee432183f,2024-12-12T00:00:00Z,2024-12-11,NaN,NaN,NaN,under
25316,25316,25316,Karl-Anthony Towns,player_points,underdog,Over,25.5,-137,New York Knicks,Atlanta Hawks,0667c1e4b0ffd20ab6acdceee432183f,2024-12-12T00:00:00Z,2024-12-11,NaN,NaN,NaN,over
25317,25317,25317,Karl-Anthony Towns,player_points,underdog,Under,25.5,-137,New York Knicks,Atlanta Hawks,0667c1e4b0ffd20ab6acdceee432183f,2024-12-12T00:00:00Z,2024-12-11,NaN,NaN,NaN,under


## Best EVs for Single Bets from draftkings, fanduel, prizepicks, and underdog

In [4]:
final_results = single_bet(filterData, odds, model, games, features, espnDate, stake=100, simulations=10000).sort_values(by='EV%', ascending=False).reset_index(drop=True)

print("\nTop 10 highest EV bets across point props:")
final_results.head(10)

Processing single bets...

DEBUG - Jalen Johnson:
  Odds: -115 (under)
  Line: 23.5
  Prediction: 16.71
  Std Dev: 7.180219742846005
  Prob Over: 0.173
  Decimal Odds: 1.87
  Breakeven: 0.535
  Simulated Mean: 16.90
  Model Prediction: 16.71

DEBUG - Jalen Johnson:
  Odds: -158 (under)
  Line: 21.5
  Prediction: 16.71
  Std Dev: 7.180219742846005
  Prob Over: 0.250
  Decimal Odds: 1.63
  Breakeven: 0.612
  Simulated Mean: 16.83
  Model Prediction: 16.71

DEBUG - Josh Hart:
  Odds: 114 (under)
  Line: 19.5
  Prediction: 12.08
  Std Dev: 2.936362072739365
  Prob Over: 0.007
  Decimal Odds: 2.14
  Breakeven: 0.467
  Simulated Mean: 12.04
  Model Prediction: 12.08

DEBUG - OG Anunoby:
  Odds: 105 (over)
  Line: 15.5
  Prediction: 16.48
  Std Dev: 6.43255608430877
  Prob Over: 0.558
  Decimal Odds: 2.05
  Breakeven: 0.488
  Simulated Mean: 16.52
  Model Prediction: 16.48

DEBUG - Josh Hart:
  Odds: -145 (over)
  Line: 19.5
  Prediction: 12.08
  Std Dev: 2.936362072739365
  Prob Over: 0.005


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
0,Josh Hart,draftkings,points,19.5,114,under,12.08,0.007,0.993,0.467,112.57,0.99,0.49,0.25,"(6.3, 18.0)"
1,Jalen Brunson,espnbet,points,14.5,110,over,23.67,0.919,0.081,0.476,93.01,0.85,0.42,0.21,"(10.5, 36.5)"
2,Alperen Sengun,draftkings,points,26.5,124,under,19.29,0.144,0.856,0.446,91.65,0.74,0.37,0.18,"(6.4, 32.8)"
3,Jalen Brunson,fanduel,points,13.5,-114,over,23.67,0.941,0.059,0.533,76.70,0.87,0.44,0.22,"(10.7, 36.8)"
4,Fred VanVleet,draftkings,points,6.5,-130,over,17.90,0.983,0.017,0.565,73.84,0.96,0.48,0.24,"(7.4, 28.5)"
5,Jalen Brunson,draftkings,points,15.5,-110,over,23.67,0.889,0.111,0.524,69.66,0.77,0.38,0.19,"(10.6, 36.9)"
6,Bogdan Bogdanović,draftkings,points,8.5,114,over,11.92,0.752,0.248,0.467,60.93,0.53,0.27,0.13,"(2.2, 22.5)"
7,De'Andre Hunter,draftkings,points,20.5,105,under,15.83,0.225,0.775,0.488,58.81,0.56,0.28,0.14,"(3.9, 28.4)"
8,Zaccharie Risacher,fanduel,points,3.5,-136,over,11.57,0.908,0.092,0.576,57.56,0.78,0.39,0.20,"(1.0, 29.7)"
9,Jalen Johnson,draftkings,points,23.5,-115,under,16.71,0.173,0.827,0.535,54.67,0.63,0.31,0.16,"(3.6, 31.0)"


## Best EVs for 2-leg parlays on prizepicks, underdogs or fanduel

In [9]:
results = prizepickspairsEV(filterData, odds, model, games, features, espnDate, stake=100, simulations=10000).sort_values(by='EV%', ascending=False).reset_index(drop=True)
print("\nTop 10 highest EV bets:")
results.head()

Processing PrizePicks pairs...

Top 10 highest EV bets:


,PLAYER 1,CATEGORY 1,LINE 1,SIDE 1,PREDICTION 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,LINE 2,SIDE 2,PREDICTION 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,TYPE,PROBABILITY,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER
0,Josh Hart,points,19.5,under,12.08,0.005,0.995,"(6.3, 18.0)",Fred VanVleet,points,6.5,over,17.90,0.985,0.015,"(7.7, 28.5)",UNDER/OVER,0.9797,1.939,0.97,0.48,0.24
1,Josh Hart,points,19.5,under,12.08,0.005,0.995,"(6.3, 18.0)",Zaccharie Risacher,points,3.5,over,11.57,0.906,0.094,"(1.2, 29.8)",UNDER/OVER,0.9014,1.704,0.85,0.43,0.21
2,Zaccharie Risacher,points,3.5,over,11.57,0.906,0.094,"(1.2, 29.8)",Fred VanVleet,points,6.5,over,17.90,0.985,0.015,"(7.7, 28.5)",OVER/OVER,0.8929,1.679,0.84,0.42,0.21
3,Josh Hart,points,19.5,under,12.08,0.005,0.995,"(6.3, 18.0)",Alperen Sengun,points,26.5,under,19.29,0.148,0.852,"(6.2, 32.7)",UNDER/UNDER,0.8476,1.543,0.77,0.39,0.19
4,Jalen Johnson,points,23.5,under,16.71,0.173,0.827,"(3.8, 30.6)",Josh Hart,points,19.5,under,12.08,0.005,0.995,"(6.3, 18.0)",UNDER/UNDER,0.8227,1.468,0.73,0.37,0.18
